In [ ]:
# 最新 0.8860
import os
import numpy as np
# 从 GIP 模块导入所有内容
from GIP import *
from model import *
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
import pickle
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import matplotlib
import matplotlib.pyplot as plt
from utils import *
# 使用 M4 芯片的 GPU (MPS) - 优化配置
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"✅ 使用 MPS (GPU) 设备: {device}")
    print(f"   MPS 后端已启用，将使用 M4 芯片的 GPU 加速")
    # MPS 特定优化：确保使用 MPS 后端（禁用 CPU fallback）
    os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '0')
else:
    device = torch.device("cpu")
    print(f"⚠️ MPS 不可用，使用 CPU 设备: {device}")
    print(f"   提示：请确保 PyTorch 版本支持 MPS（需要 PyTorch >= 1.12）")
seed_everything(42)
path = "./scores/"
if not os.path.exists(path):
    os.makedirs(path)
print("11116")
# load adj, sim
# Load adjacent matrix(data_processing.py)
adj_np = pd.read_csv(r"rd_adj.csv", index_col=0).values
# Load piRNA similarity based on Smith-Waterman method(gen_half_p2p_simth.py)
# 读取包含索引的CSV文件
p_sim_df = pd.read_csv(r"cosine_similarity_matrix.csv", index_col=0)

# 将DataFrame转换为NumPy数组
p_sim_np = p_sim_df.values

# 打印数组的形状
print("Shape of p_sim_np:", p_sim_np.shape)
# Load piRNA feature based on word2vec method(gen_pfeat_gensim.py)
gensim_feat = np.load(
    r"gensim_feat_128.npy",
    allow_pickle=True,
).flat[0]
p_kmers_emb = gensim_feat["p_kmers_emb"]
pad_kmers_id_seq = gensim_feat["pad_kmers_id_seq"]
# Load disease similarity based on DO DAG(gen_d2d_do.py)
d_sim_np = pd.read_csv(r"d2d_do.csv", index_col=0).values
d_feat = d_sim_np

num_c, num_d = adj_np.shape


p_sim = torch.FloatTensor(p_sim_np).to(device)
d_sim = torch.FloatTensor(d_sim_np).to(device)
adj = torch.FloatTensor(adj_np).to(device)
p_kmers_emb = torch.FloatTensor(p_kmers_emb).to(device)
pad_kmers_id_seq = torch.tensor(pad_kmers_id_seq).to(device)
d_feat = torch.FloatTensor(d_feat).to(device)

k = 1
merge_win_size = 32
context_size_list = [1, 3, 5]
dll_out_size = 128 * len(context_size_list) * k

# 回到原始稳定配置（128维度，这是最稳定的配置）
gcn_out_dim = 128 * k
gcn_hidden_dim = 128 * k
num_layers, dropout = 1, 0.7 # 回到原始dropout
query_size = key_size = 128 * k
value_size = 128 * k
enc_ffn_num_hiddens, n_enc_heads = 128, 2* k  # 回到原始配置

lr, num_epochs = 0.02, 500 # 回到原始学习率和训练轮数
# 降低学习率，防止后期训练不稳定导致AUC下降

feat_init_d = d_feat.shape[1]


class MaskedBCELoss(nn.BCELoss):
    def forward(self, pred, adj, train_mask, test_mask):
        self.reduction = "none"
        unweighted_loss = super(MaskedBCELoss, self).forward(pred, adj)
        train_loss = (unweighted_loss * train_mask).sum()
        test_loss = (unweighted_loss * test_mask).sum()
        return train_loss, test_loss


def grad_clipping(net, theta):
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad and p.grad is not None]
    else:
        params = [p for p in net.params if p.grad is not None]
    if len(params) == 0:
        return
    norm = torch.sqrt(sum(torch.sum((p.grad**2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm


def fit(
    fold_cnt,
    model,
    adj,
    adj_full,
    pad_kmers_id_seq,
    d_feat,
    train_mask,
    test_mask,
    lr,
    num_epochs,
    pos_train_ij=None,
    rn_ij=None,
    use_pairwise=False,
    pairwise_weight=0.5,
    num_pairs=20000,
):
    def xavier_init_weights(m):
        if type(m) == nn.Linear:
            nn.init.xavier_uniform_(m.weight)

    model.apply(xavier_init_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Warmup机制：前10个epoch逐渐增加学习率，提高训练稳定性
    warmup_epochs = 0
    warmup_factor = 0.1  # 从10%的学习率开始
    
    # 设置最低学习率：降到这个值之后一直维持，不再继续下降
    min_lr = 0.0000001
    
    # 记录最佳AUC，用于基于AUC的学习率调整（不进行早停）
    best_auc = 0.0
    best_auc_epoch = 0
    best_model_state = None
    
    # AUC下降检测：当AUC明显下降时，恢复最佳模型并降低学习率
    auc_drop_threshold = 0.01  # AUC下降超过0.001就触发恢复
    consecutive_drops = 1  # 连续下降次数
    max_consecutive_drops = 1  # 连续1次下降就恢复模型
    
    loss = MaskedBCELoss()

    def pairwise_auc_loss(scores, pos_idx_np, neg_idx_np, sample_pairs=20000):
        # 采样若干正负对，最大化 s_pos - s_neg（优化AUC到0.92）
        if len(pos_idx_np) == 0 or len(neg_idx_np) == 0:
            return torch.tensor(0.0, device=scores.device)
        import numpy as np
        n_pos = pos_idx_np.shape[0]
        n_neg = neg_idx_np.shape[0]
        sp = min(sample_pairs, n_pos * n_neg)
        pos_sel = np.random.randint(0, n_pos, size=sp)
        neg_sel = np.random.randint(0, n_neg, size=sp)
        pos_pairs = pos_idx_np[pos_sel]
        neg_pairs = neg_idx_np[neg_sel]
        s_pos = scores[tuple(pos_pairs.T)]
        s_neg = scores[tuple(neg_pairs.T)]
        
        # 使用简单的logsigmoid loss，最大化 s_pos - s_neg（稳定且有效）
        diff = s_pos - s_neg
        return -torch.nn.functional.logsigmoid(diff).mean()

    test_idx = torch.argwhere(test_mask == 1)
    # test_idx = torch.argwhere(torch.ones_like(test_mask) == 1)
    
    for epoch in range(num_epochs):
        # Warmup机制：前warmup_epochs个epoch逐渐增加学习率
        if epoch < warmup_epochs:
            warmup_lr = lr * (warmup_factor + (1 - warmup_factor) * (epoch + 1) / warmup_epochs)
            # 确保warmup阶段的学习率不低于最低学习率
            warmup_lr = max(warmup_lr, min_lr)
            for param_group in optimizer.param_groups:
                param_group['lr'] = warmup_lr
        
        model.train()
        optimizer.zero_grad()
        pred = model(pad_kmers_id_seq, d_feat, adj_full)
        train_loss, test_loss = loss(pred, adj, train_mask, test_mask)

        if use_pairwise and pos_train_ij is not None and rn_ij is not None:
            pw_loss = pairwise_auc_loss(pred, pos_train_ij, rn_ij, sample_pairs=num_pairs)
            total_loss = (1 - pairwise_weight) * train_loss + pairwise_weight * pw_loss
        else:
            total_loss = train_loss

        total_loss.backward()
        
        # 梯度裁剪：防止梯度爆炸（降低阈值，更严格的控制）
        # grad_clipping(model, 3)
        
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred = model(pad_kmers_id_seq, d_feat, adj_full)

        scores = pred[tuple(list(test_idx.T))].cpu().detach().numpy()
        np.save(rf"./scores/f{fold_cnt}_e{epoch}_scores.npy", scores)
        
        # 计算AUC用于监控和基于AUC的学习率调整
        from sklearn.metrics import roc_auc_score
        labels = adj[tuple(list(test_idx.T))].cpu().detach().numpy()
        current_auc = roc_auc_score(labels, scores)
        
        logger.update(
            fold_cnt, epoch, adj, pred, test_idx, train_loss.item(), test_loss.item()
        )
        
        # 更新最佳模型，并根据AUC下降调整学习率（不进行早停）
        if current_auc > best_auc:
            best_auc = current_auc
            best_auc_epoch = epoch
            consecutive_drops = 0  # 重置连续下降计数
            # 保存最佳模型状态
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if epoch % 10 == 0 or epoch < 20:
                print(f"  Epoch {epoch}: 新的最佳AUC = {best_auc:.4f}")
        else:
            # 在warmup之后，检测AUC是否明显下降
            if epoch >= warmup_epochs:
                # AUC下降检测：检测AUC是否明显下降
                auc_drop = best_auc - current_auc
                if auc_drop > auc_drop_threshold:
                    consecutive_drops += 1
                    print(f"  Epoch {epoch}: AUC下降 {auc_drop:.4f} (当前={current_auc:.4f}, 最佳={best_auc:.4f}), 连续下降{consecutive_drops}次")
                    
                    # 如果连续下降超过阈值，恢复最佳模型并降低学习率
                    if consecutive_drops >= max_consecutive_drops:
                        print(f"  Epoch {epoch}: AUC连续下降{consecutive_drops}次，恢复最佳模型@epoch{best_auc_epoch} (AUC={best_auc:.4f})")
                        if best_model_state is not None:
                            model.load_state_dict(best_model_state)
                            # 降低学习率，但不会低于最低学习率
                            for param_group in optimizer.param_groups:
                                old_lr = param_group['lr']
                                new_lr = old_lr * 0.8
                                # 确保学习率不会低于最低学习率
                                new_lr = max(new_lr, min_lr)
                                param_group['lr'] = new_lr
                                if new_lr > min_lr:
                                    print(f"    学习率从 {old_lr:.6f} 降低到 {new_lr:.6f}")
                                else:
                                    print(f"    学习率从 {old_lr:.6f} 降低到 {new_lr:.6f} (已到达最低学习率 {min_lr:.6f}，将维持不变)")
                        consecutive_drops = 0  # 重置计数
                else:
                    consecutive_drops = 0  # 如果下降不明显，重置计数
        

logger = Logger(10)  # 10个fold

with open(r"fold_info.pickle", "rb") as f:
    fold_info = pickle.load(f)
with open("rn_ij_list_pu.pickle", "rb") as f:
    rn_ij_list_pu = pickle.load(f)
# with open(rf"rn_ij_list_spy.pickle", "rb") as f:
#     rn_ij_list_spy = pickle.load(f)
with open(rf"rn_ij_list_two.pickle","rb") as f :
    rn_ij_list_two = pickle.load(f)


pos_train_ij_list = fold_info["pos_train_ij_list"]
pos_test_ij_list = fold_info["pos_test_ij_list"]
unlabelled_train_ij_list = fold_info["unlabelled_train_ij_list"]
unlabelled_test_ij_list = fold_info["unlabelled_test_ij_list"]
p_gip_list = fold_info["c_gip_list"]
d_gip_list = fold_info["d_gip_list"]
# 运行所有 fold (0-9)，所有fold使用统一配置
for i in range(10):
    print(f"fold {i}")
    pos_train_ij = pos_train_ij_list[i]
    pos_test_ij = pos_test_ij_list[i]
    unlabelled_train_ij = unlabelled_train_ij_list[i]
    unlabelled_test_ij = unlabelled_test_ij_list[i]
    p_gip = p_gip_list[i]
    d_gip = d_gip_list[i]
    # 使用two+spy组合负样本
    # rn_ij = np.concatenate((rn_ij_list_two[i], rn_ij_list_spy[i]))
    # rn_ij = rn_ij_list_spy[i]
    rn_ij = rn_ij_list_two[i]
    # rn_ij = rn_ij_list_pu[i]
    A_corner_np = np.zeros_like(adj_np)
    A_corner_np[tuple(list(pos_train_ij.T))] = 1

    A_np = np.concatenate(
        (
            np.concatenate(((p_sim_np + p_gip) / 2, A_corner_np), axis=1),
            np.concatenate(((A_corner_np).T, (d_sim_np + d_gip) / 2), axis=1),
        ),
        axis=0,
    )

    # train_mask_np = np.ones_like(adj_np)
    train_mask_np = np.zeros_like(adj_np)
    train_mask_np[tuple(list(pos_train_ij.T))] = 1
    # train_mask_np[tuple(list(unlabelled_train_ij.T))] = 1
    train_mask_np[tuple(list(rn_ij.T))] = 1

    test_mask_np = np.zeros_like(adj_np)
    test_mask_np[tuple(list(pos_test_ij.T))] = 1
    test_mask_np[tuple(list(unlabelled_test_ij.T))] = 1
    num_positive_samples = np.sum(train_mask_np == 1)
    num_negative_samples = np.sum(train_mask_np == 0)
    print(f"Number of positive samples: {num_positive_samples}")
    print(f"Number of negative samples: {num_negative_samples}")


    A_corner = torch.FloatTensor(A_corner_np).to(device)
    A = torch.FloatTensor(A_np).to(device)
    train_mask = torch.FloatTensor(train_mask_np).to(device)
    test_mask = torch.FloatTensor(test_mask_np).to(device)

    # 清空缓存（根据设备类型选择）
    if device.type == "mps":
        torch.mps.empty_cache()
    elif device.type == "cuda":
        torch.cuda.empty_cache()
    deep_lnc_loc = DeepLncLoc(
        p_kmers_emb, dropout, merge_win_size, context_size_list, dll_out_size
    ).to(device)

    # 切换到简化版GraphSAGE（基于GCN成功经验优化）
    gcn = GraphSAGE(
        p_feat_dim=dll_out_size,
        d_feat_dim=feat_init_d,
        n_hidden=gcn_hidden_dim,
        dropout=dropout,
    ).to(device)

    p_encoder = TransformerEncoder(
        q_in_dim=gcn_out_dim,
        kv_in_dim=gcn_out_dim,
        key_size=key_size,
        query_size=query_size,
        value_size=value_size,
        ffn_num_hiddens=enc_ffn_num_hiddens,
        num_heads=n_enc_heads,
        num_layers=num_layers,
        dropout=dropout,
        bias=False,
    ).to(device)

    d_encoder = TransformerEncoder(
        q_in_dim=gcn_out_dim,
        kv_in_dim=gcn_out_dim,
        key_size=key_size,
        query_size=query_size,
        value_size=value_size,
        ffn_num_hiddens=enc_ffn_num_hiddens,
        num_heads=n_enc_heads,
        num_layers=num_layers,
        dropout=dropout,
        bias=False,
    ).to(device)

    predictor = Predictor().to(device)

    # 所有折都启用pairwise loss，强化排序（关键优化：提升AUC到0.92，舍弃AUPR）
    use_pairwise_flag = False  # 启用所有fold的pairwise loss，这是提升AUC的关键
    
    # 统一配置：所有fold使用相同的参数配置
    pairwise_weight = 0.2 # 统一的pairwise权重
    num_pairs_fold = 40000  # 统一的采样对数量
    model = PUTransGCN(deep_lnc_loc, gcn, p_encoder, d_encoder, predictor).to(device)
    fit(
        i,
        model,
        adj,
        A,
        pad_kmers_id_seq,
        d_feat,
        train_mask,
        test_mask,
        lr,
        num_epochs,
        pos_train_ij=pos_train_ij,
        rn_ij=rn_ij,
        use_pairwise=use_pairwise_flag,
        pairwise_weight=pairwise_weight,
        num_pairs=num_pairs_fold,  # 统一的采样对数量
    )
    # 获取最大已分配内存（根据设备类型选择）
    if device.type == "cuda":
        max_allocated_memory = torch.cuda.max_memory_allocated()
        print(f"最大已分配内存量: {max_allocated_memory / 1024 ** 2} MB")
    elif device.type == "mps":
        # MPS 不支持 max_memory_allocated，使用其他方式监控
        print("MPS 设备：内存使用情况可通过系统监控工具查看")
    else:
        print("CPU 设备：无需监控 GPU 内存")
logger.save("circRNA_result")


✅ 使用 MPS (GPU) 设备: mps
   MPS 后端已启用，将使用 M4 芯片的 GPU 加速
11116
Shape of p_sim_np: (533, 533)
fold 0
Number of positive samples: 17387
Number of negative samples: 30050
fold:0, epoch:0, f1: 0.090909, f2: 0.133197, rank_idx: 0.389979, auc: 0.611533, aupr: 0.024527, acc: 0.940569, specificity: 0.94984, threshold: 0.007088, recall: 0.216667, precision: 0.052419, mcc: 0.083594, train_loss: 361319, test_loss: 112779
  Epoch 0: 新的最佳AUC = 0.6115
fold:0, epoch:1, f1: 0.060109, f2: 0.100733, rank_idx: 0.577243, auc: 0.421873, aupr: 0.01517, acc: 0.927503, specificity: 0.937033, threshold: 0.0, recall: 0.183333, precision: 0.035948, mcc: 0.054756, train_loss: 609985, test_loss: 186971
  Epoch 1: AUC下降 0.1897 (当前=0.4219, 最佳=0.6115), 连续下降1次
  Epoch 1: AUC连续下降1次，恢复最佳模型@epoch0 (AUC=0.6115)
    学习率从 0.020000 降低到 0.016000
fold:0, epoch:2, f1: 0.056701, f2: 0.102041, rank_idx: 0.592894, auc: 0.406019, aupr: 0.014711, acc: 0.879452, specificity: 0.8873, threshold: 0.0, recall: 0.266667, precision: 0.029412,

/Users/mac/Desktop/circR2disease/utils.py:184: RuntimeWarning: invalid value encountered in scalar divide
  mcc = (tp * tn - fp * fn) / np.sqrt(


fold:3, epoch:11, f1: 0.038278, f2: 0.060193, rank_idx: 0.499259, auc: 0.517122, aupr: 0.019504, acc: 0.012648, specificity: 0.0, threshold: 0.0, recall: 1.0, precision: 0.012648, mcc: nan, train_loss: 136286, test_loss: 59793
  Epoch 11: AUC下降 0.4187 (当前=0.5171, 最佳=0.9358), 连续下降1次
  Epoch 11: AUC连续下降1次，恢复最佳模型@epoch8 (AUC=0.9358)
    学习率从 0.008192 降低到 0.006554


/Users/mac/Desktop/circR2disease/utils.py:184: RuntimeWarning: invalid value encountered in scalar divide
  mcc = (tp * tn - fp * fn) / np.sqrt(


fold:3, epoch:12, f1: 0.037383, f2: 0.060193, rank_idx: 0.502582, auc: 0.509853, aupr: 0.016931, acc: 0.012648, specificity: 0.0, threshold: 0.0, recall: 1.0, precision: 0.012648, mcc: nan, train_loss: 132041, test_loss: 55186
  Epoch 12: AUC下降 0.4260 (当前=0.5099, 最佳=0.9358), 连续下降1次
  Epoch 12: AUC连续下降1次，恢复最佳模型@epoch8 (AUC=0.9358)
    学习率从 0.006554 降低到 0.005243


...

fold:3, epoch:39, f1: 0.038835, f2: 0.060193, rank_idx: 0.505695, auc: 0.51271, aupr: 0.015866, acc: 0.012648, specificity: 0.0, threshold: 0.0, recall: 1.0, precision: 0.012648, mcc: nan, train_loss: 125674, test_loss: 49976
  Epoch 39: AUC下降 0.4339 (当前=0.5127, 最佳=0.9466), 连续下降1次
  Epoch 39: AUC连续下降1次，恢复最佳模型@epoch38 (AUC=0.9466)
    学习率从 0.000148 降低到 0.000118


/Users/mac/Desktop/circR2disease/utils.py:184: RuntimeWarning: invalid value encountered in scalar divide
  mcc = (tp * tn - fp * fn) / np.sqrt(


fold:3, epoch:40, f1: 0.038835, f2: 0.060193, rank_idx: 0.506433, auc: 0.511438, aupr: 0.015505, acc: 0.012648, specificity: 0.0, threshold: 0.0, recall: 1.0, precision: 0.012648, mcc: nan, train_loss: 115524, test_loss: 41840
  Epoch 40: AUC下降 0.4352 (当前=0.5114, 最佳=0.9466), 连续下降1次
  Epoch 40: AUC连续下降1次，恢复最佳模型@epoch38 (AUC=0.9466)
    学习率从 0.000118 降低到 0.000094


/Users/mac/Desktop/circR2disease/utils.py:184: RuntimeWarning: invalid value encountered in scalar divide
  mcc = (tp * tn - fp * fn) / np.sqrt(


fold:3, epoch:41, f1: 0.038835, f2: 0.060193, rank_idx: 0.501637, auc: 0.517602, aupr: 0.016563, acc: 0.012648, specificity: 0.0, threshold: 0.0, recall: 1.0, precision: 0.012648, mcc: nan, train_loss: 115708, test_loss: 44348
  Epoch 41: AUC下降 0.4290 (当前=0.5176, 最佳=0.9466), 连续下降1次
  Epoch 41: AUC连续下降1次，恢复最佳模型@epoch38 (AUC=0.9466)
    学习率从 0.000094 降低到 0.000076
fold:3, epoch:42, f1: 0.038835, f2: 0.063291, rank_idx: 0.493441, auc: 0.52426, aupr: 0.017663, acc: 0.924325, specificity: 0.934671, threshold: 0.0, recall: 0.116667, precision: 0.022364, mcc: 0.02311, train_loss: 166165, test_loss: 61862
  Epoch 42: AUC下降 0.4224 (当前=0.5243, 最佳=0.9466), 连续下降1次
  Epoch 42: AUC连续下降1次，恢复最佳模型@epoch38 (AUC=0.9466)
    学习率从 0.000076 降低到 0.000060
fold:3, epoch:43, f1: 0.044554, f2: 0.077055, rank_idx: 0.47343, auc: 0.53779, aupr: 0.019714, acc: 0.918634, specificity: 0.92848, threshold: 0.0, recall: 0.15, precision: 0.026163, mcc: 0.033817, train_loss: 139775, test_loss: 54725
  Epoch 43: AUC下降 0.4088

In [6]:
import pandas as pd

# 1️⃣ 读取修改后的文件
file = "circRNA_result.xlsx"
xls = pd.ExcelFile(file)

# 2️⃣ 存储每个fold的结果
results = []

# 3️⃣ 遍历每个sheet（fold0 ~ fold4）
for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet)
    
    # 找出列名中包含AUC和AUPR的列（不区分大小写）
    auc_col = None
    aupr_col = None
    for col in df.columns:
        if "auc" in col.lower():
            auc_col = col
        if "aupr" in col.lower():
            aupr_col = col

    if auc_col and aupr_col:
        auc_mean = df[auc_col].mean()
        auc_max = df[auc_col].max()  # 添加AUC最高值
        aupr_mean = df[aupr_col].mean()
        print(f"✅ {sheet}: AUC平均值={auc_mean:.4f}, AUC最高值={auc_max:.4f}, AUPR平均值={aupr_mean:.4f}")
        results.append({
            "fold": sheet, 
            "AUC_mean": auc_mean, 
            "AUC_max": auc_max,  # 添加AUC最高值
            "AUPR_mean": aupr_mean
        })
    else:
        print(f"⚠️ {sheet} 未找到 AUC 或 AUPR 列。")

# 4️⃣ 转为 DataFrame 汇总
res_df = pd.DataFrame(results)

# 5️⃣ 计算整体平均
overall_auc = res_df["AUC_mean"].mean()
overall_auc_max = res_df["AUC_max"].mean()  # 所有fold的AUC最高值的平均
overall_aupr = res_df["AUPR_mean"].mean()

print("\n📊 各fold统计：")
print(res_df)

print(f"\n🌟 全部fold整体平均：")
print(f"   AUC平均值: {overall_auc:.4f}")
print(f"   AUC最高值平均: {overall_auc_max:.4f}")
print(f"   AUPR平均值: {overall_aupr:.4f}")

# 可选：保存结果
res_df.to_excel("circRNA_result_summary.xlsx", index=False)
print(f"\n✅ 结果已保存到 circRNA_result_summary.xlsx")



✅ fold0: AUC平均值=0.6193, AUC最高值=0.6270, AUPR平均值=0.0235
✅ fold1: AUC平均值=0.8716, AUC最高值=0.8762, AUPR平均值=0.1699
✅ fold2: AUC平均值=0.8794, AUC最高值=0.8870, AUPR平均值=0.1281
✅ fold3: AUC平均值=0.9177, AUC最高值=0.9466, AUPR平均值=0.2418
✅ fold4: AUC平均值=0.8862, AUC最高值=0.9138, AUPR平均值=0.0851
✅ fold5: AUC平均值=0.8350, AUC最高值=0.8502, AUPR平均值=0.0522
✅ fold6: AUC平均值=0.9108, AUC最高值=0.9387, AUPR平均值=0.0943
✅ fold7: AUC平均值=0.8749, AUC最高值=0.8867, AUPR平均值=0.2006
✅ fold8: AUC平均值=0.8324, AUC最高值=0.8390, AUPR平均值=0.1286
✅ fold9: AUC平均值=0.8634, AUC最高值=0.8718, AUPR平均值=0.1556

📊 各fold统计：
    fold  AUC_mean   AUC_max  AUPR_mean
0  fold0  0.619268  0.627030   0.023487
1  fold1  0.871599  0.876201   0.169894
2  fold2  0.879424  0.887029   0.128090
3  fold3  0.917672  0.946632   0.241791
4  fold4  0.886171  0.913843   0.085148
5  fold5  0.834992  0.850222   0.052213
6  fold6  0.910846  0.938744   0.094349
7  fold7  0.874898  0.886657   0.200621
8  fold8  0.832409  0.838961   0.128627
9  fold9  0.863450  0.871780   0.155580

🌟 全部fol